# Dynamic Treatment on Networks — Q-Ising
**ArXivist-generated reproduction notebook**  
Paper: https://arxiv.org/abs/2605.06564  
Authors: Bengusu Nar, Jiguang Li, Veronika Ročková, Panos Toulis  
Generated: 2026-05-09

This notebook walks through all three stages of the Q-Ising pipeline, runs a small-scale training loop on a synthetic SBM network, and compares results against the paper's reported benchmarks.

In [ ]:
# Cell 1 — Environment Check
import sys
print(f'Python: {sys.version}')

try:
    import torch
    print(f'PyTorch: {torch.__version__}')
    print(f'CUDA available: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'GPU: {torch.cuda.get_device_name(0)}')
        print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    else:
        print('Running on CPU — experiments run on M1 CPU in the paper, so this is fine')
except ImportError:
    print('PyTorch not installed — run Cell 2 first')

try:
    import numpy as np
    print(f'NumPy: {np.__version__}')
except ImportError:
    print('NumPy not installed')

In [ ]:
# Cell 2 — Install Q-Ising in editable mode (run once)
import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', '..'],
    capture_output=True, text=True, cwd='..'
)
if result.returncode == 0:
    print('Installation successful')
    print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
else:
    print('Installation failed:')
    print(result.stderr[-1000:])

In [ ]:
# Cell 2b — Add src to path
import sys, os
from pathlib import Path

REPO_ROOT = Path('..').resolve()
SRC_PATH = REPO_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))
print(f'Repo root: {REPO_ROOT}')
print(f'Source path: {SRC_PATH}')

## Paper Overview

### Problem
When a planner applies treatments on a social network (e.g. seeding a product, vaccinating nodes), they face two decisions: **whom** to treat and **when**. Treating an influential node early can trigger cascades that change which nodes are worth targeting next. Standard treatment frameworks ignore either the network structure (dynamic treatment regimes) or the time dimension (static influence maximization).

### Key Innovation: Q-Ising
Q-Ising is a 3-stage pipeline that bridges Bayesian network inference and offline reinforcement learning:

| Stage | Method | Paper Section |
|-------|--------|---------------|
| 1 — Ising Inference | Bayesian dynamic Ising model estimated via EMVS or MCMC | §3.1 |
| 2 — Offline RL | Conservative Q-Learning (CQL) over latent bin-level states | §3.2 |
| 3 — Ensemble Policy | Majority-vote across MCMC posterior draws | §3.3 |

### Core State Construction
The Q-Ising state at period $t$ is (Eq. 8):

$$s_t = (\bar{l}^0_t,\ \bar{y}_{t-1}) \in [0,1]^K \times [0,1]^K$$

where $\bar{l}^0_{t,k}$ is the **counterfactual** (no-intervention) adoption probability averaged over bin $k$, and $\bar{y}_{t-1,k}$ is the observed bin-level adoption. This 2K-dimensional state feeds into CQL.

### Key Theoretical Result
Theorem 2 gives a finite-sample regret bound:

$$\text{Reg}_H(\hat{\pi}^\circ) \leq 2\beta\, U^\mu_D(\pi^\star_S) + C\left[\Delta_{\text{abs}} + H\varepsilon_{\text{lin}} + HL_{\text{st}}\varepsilon_\theta\right]$$

decomposing into offline-RL uncertainty, network abstraction error, and Ising estimation error.

## Component 1: Network & SIS Simulator

We build a small Stochastic Block Model (SBM) network and simulate SIS dynamics on it.
This corresponds to the experiment in **Section 5.1** of the paper.

SIS dynamics (Appendix E.1) follow three ordered sub-steps each period:
1. **Churn**: each adopted node $i$ reverts with probability $\delta_i$
2. **Seed**: selected node is force-adopted (perfect treatment)
3. **Spread**: each adopted $i$ infects susceptible neighbor $j$ with probability $\beta_i$:

$$p^{\text{adopt}}_j = 1 - \prod_{i \in N(j),\, i\text{ adopted}} (1 - \beta_i)$$

In [ ]:
# Cell 4 — Build a small SBM network for demonstration
import numpy as np
from q_ising.utils.config import set_global_seed
from q_ising.utils.sbm_generator import generate_sbm, get_sbm_block_labels
from q_ising.data.network import NetworkData

set_global_seed(42)

# Mini SBM: 60 nodes, 4 blocks (scaled down from paper's 500 for speed)
N_PER_BLOCK = [22, 22, 8, 8]   # paper uses [187, 187, 63, 63]
N = sum(N_PER_BLOCK)
K = 4

M = generate_sbm(n_per_block=N_PER_BLOCK, p_in=0.1, p_out=0.01, seed=42)
bin_labels = get_sbm_block_labels(N_PER_BLOCK)
network = NetworkData(M=M, bin_labels=bin_labels)

print(network)
print(f'Adjacency matrix shape: {M.shape}')
print(f'Number of edges: {M.sum() // 2}')
print(f'Bin sizes: {[len(network.get_bin_members(k)) for k in range(K)]}')
print(f'Mean degree: {M.sum(axis=1).mean():.2f}')

In [ ]:
# Cell 5 — SIS Simulator: generate a short observational panel
from q_ising.data.sis_simulator import SISSimulator
from q_ising.evaluation.baselines import RandomPolicy

# Mini spread/churn rates (same structure as paper's Section 5.1)
SPREAD_RATES = [0.010, 0.012, 0.10, 0.12]
CHURN_RATES  = [0.40,  0.40,  0.20, 0.20]

simulator = SISSimulator(
    network=network,
    spread_rates=SPREAD_RATES,
    churn_rates=CHURN_RATES,
    spread_by_bin=True,
)

# Generate a short training panel using random policy (T=30 for demo; paper uses T=100)
T_TRAIN = 30
random_policy = RandomPolicy(network)
panel = simulator.generate_panel(T=T_TRAIN, policy=random_policy, seed=42)

print(panel)
print(f'Mean adoption rate in panel: {panel.outcomes.mean():.4f}')
print(f'Actions (first 10): {panel.actions[:10]}')
print(f'Outcome shape: {panel.outcomes.shape}  [T x N]')

## Component 2: Dynamic Ising Model (Stage 1)

The linear predictor for node $i$ in bin $B_k$ at time $t$ is **(Section 3.1, Eq. 4)**:

$$\eta_{i,t}(a_t;\theta_i) = \beta_{0,k} + \beta_{1,k}\mathbf{1}_{a_t=i} + \beta_{2,k}y_{i,t-1} + \beta_{3,k}\mathbf{1}_{a_t\in N_i} + \sum_{j\in N_i}\gamma_{k,m_j}y_{j,t-1}$$

Parameters:
- $\beta_{1,k}$: **direct treatment effect** for bin $k$
- $\beta_{2,k}$: **persistence** of past adoption
- $\beta_{3,k}$: **spillover** from treating a neighbor
- $\gamma_{k,m_j}$: **peer influence** from neighbor $j$'s bin to node $i$'s bin

Coupling parameters use a **spike-and-slab prior** (Section 3.1):
$$\gamma_{i,j}|z_{ij} \sim (1-z_{ij})\mathcal{N}(0, v_0) + z_{ij}\mathcal{N}(0, v_1), \quad v_0=0.01,\ v_1=10$$

In [ ]:
# Cell 6 — Stage 1: Fit the Dynamic Ising Model via EMVS
from q_ising.models.ising import DynamicIsingModel
from q_ising.utils.config import IsingConfig
import time

ising_cfg = IsingConfig(
    estimation_method='emvs',
    v0=0.01,    # spike variance (Section 3.1)
    v1=10.0,    # slab variance (Section 3.1)
    c=1.0,      # inclusion scale (Section 3.1)
    tau_sq=10.0 # beta prior variance (Section 3.1)
)

ising = DynamicIsingModel(network=network, config=ising_cfg)

t0 = time.time()
ising.fit_emvs(panel)
elapsed = time.time() - t0

print(f'Ising model fitted in {elapsed:.2f}s')
print(f'{ising}')

# Inspect parameters for node 0 (bin 0)
p0 = ising._params[0]
print(f'\nNode 0 (bin {p0.bin_k}) parameters:')
print(f'  beta_0 (intercept):  {p0.beta_0:.4f}')
print(f'  beta_1 (treatment):  {p0.beta_1:.4f}')
print(f'  beta_2 (persistence):{p0.beta_2:.4f}')
print(f'  beta_3 (spillover):  {p0.beta_3:.4f}')
print(f'  gamma (couplings):   {p0.gamma[:5]}...')

In [ ]:
# Cell 7 — Demo: compute adoption probabilities for node 0
# P(y_{i,t}=1 | ...) = sigmoid(eta_{i,t})  [Eq. eq_adoption_prob]

y_prev = panel.outcomes[0]   # state after period 1
node = 0

# With direct treatment (action = treat node 0)
p_treated = ising.adoption_prob(y_prev, action=node, node=node)
# Without any treatment (counterfactual)
p_counterfactual = ising.adoption_prob(y_prev, action=None, node=node)
# With neighbor treated
neighbors = network.get_neighbors(node)
p_neighbor = ising.adoption_prob(y_prev, action=neighbors[0] if neighbors else None, node=node)

print(f'Node {node} adoption probabilities:')
print(f'  P(adopt | treated directly):       {p_treated:.4f}')
print(f'  P(adopt | no intervention):        {p_counterfactual:.4f}  ← used for state construction')
print(f'  P(adopt | neighbor treated):       {p_neighbor:.4f}')
print(f'  Treatment effect (direct):         {p_treated - p_counterfactual:+.4f}')

## Component 3: State Construction (Stage 1b)

The Q-Ising state aggregates node-level counterfactual probabilities to bin-level **(Section 3.1, Eqs. 6–8)**:

$$\hat{l}^0_{i,t} = \sigma(\eta_{i,t}(\emptyset;\ \hat{\theta})) \quad \text{(set } a_t=\emptyset \text{ — no intervention)}$$

$$\bar{l}^0_{t,k} = \frac{1}{|B_k|}\sum_{i\in B_k} \hat{l}^0_{i,t}, \qquad \bar{y}_{t-1,k} = \frac{1}{|B_k|}\sum_{i\in B_k} y_{i,t-1}$$

$$s_t = (\bar{l}^0_t,\ \bar{y}_{t-1}) \in [0,1]^K \times [0,1]^K \quad \text{(dim } 2K\text{)}$$

The first $K$ components are **forward-looking** (model-based), the last $K$ are **backward-looking** (observed).

In [ ]:
# Cell 8 — State Construction
from q_ising.models.state_constructor import StateConstructor

state_ctor = StateConstructor(ising_model=ising, network=network)

# Build all states for the training panel: shape [T+1, 2K]
states = state_ctor.build_all_states(panel)

print(f'State tensor shape: {states.shape}  [T+1={panel.T+1}, 2K={2*K}]')
print(f'State at t=1: {states[0]}')
print(f'  First K={K} dims (l_bar^0, forward-looking): {states[0,:K]}')
print(f'  Last K={K} dims (y_bar, backward-looking):   {states[0,K:]}')
print(f'State range: [{states.min():.4f}, {states.max():.4f}] — should be in [0,1]')

# Show how states evolve over time
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for k in range(K):
    axes[0].plot(states[:, k], label=f'Bin {k} l_bar^0')
    axes[1].plot(states[:, K+k], label=f'Bin {k} y_bar', linestyle='--')
axes[0].set_title('Forward-looking: Counterfactual adoption prob (l_bar^0)')
axes[1].set_title('Backward-looking: Observed adoption (y_bar)')
for ax in axes:
    ax.set_xlabel('Period')
    ax.set_ylabel('Bin-level probability')
    ax.legend(fontsize=8)
    ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.savefig('../results/state_evolution.png', dpi=100, bbox_inches='tight')
plt.show()
print('State evolution plot saved to results/state_evolution.png')

## Component 4: Offline Q-Learning with CQL (Stage 2)

Conservative Q-Learning (Section 3.2) minimizes:

$$\mathcal{L}_{\text{CQL}}(Q) = \underbrace{\mathbb{E}\left[(Q(s,b) - \mathcal{T}^b\bar{Q})^2\right]}_{\text{Bellman error}} + \alpha \underbrace{\mathbb{E}\left[\log\sum_{b'}\exp Q(s,b') - Q(s,b)\right]}_{\text{conservative penalty}}$$

The Q-network takes the 2K-dimensional Q-Ising state as input and outputs Q-values for all K bin actions.

**Key hyperparameters** (Appendix E.2):
- $\alpha = 0.1$ (conservative penalty)
- $\psi = 0.8$ (discount factor)
- Hidden layers: [256, 256], Dropout: 0.3, Batch norm: True

In [ ]:
# Cell 9 — Build offline RL transitions from panel + states
transitions = panel.to_rl_transitions(states=states, bin_labels=network.bin_labels)

print(f'Number of transitions: {len(transitions)}  (T-1 = {panel.T-1})')
t0 = transitions[0]
print(f'\nExample transition (t=1):')
print(f'  state shape:      {t0.state.shape}  → {t0.state}')
print(f'  bin_action:       {t0.bin_action}  (bin index in [0, K-1])')
print(f'  reward:           {t0.reward:.4f}  (mean adoption rate)')
print(f'  next_state shape: {t0.next_state.shape}')

# Show reward distribution
rewards = [t.reward for t in transitions]
print(f'\nReward stats: mean={np.mean(rewards):.4f}, std={np.std(rewards):.4f}')
print(f'Bin action distribution: {np.bincount([t.bin_action for t in transitions])}')

In [ ]:
# Cell 10 — Stage 2: Train CQL (mini-training, 500 steps for demo)
# Paper uses 30,000 steps; we use 500 for a quick demonstration
from q_ising.training.cql_trainer import CQLTrainer
from q_ising.utils.config import CQLConfig
import time

mini_cql_cfg = CQLConfig(
    hidden_layers=[64, 64],   # reduced from [256,256] for demo speed
    learning_rate=3e-4,
    batch_size=16,
    max_steps=500,            # reduced from 30000 for demo
    steps_per_epoch=100,
    early_stopping_patience=10,
    dropout_rate=0.3,
    batch_norm=True,
    alpha=0.1,
    discount=0.8,
)

state_dim = 2 * K
trainer = CQLTrainer(K=K, state_dim=state_dim, config=mini_cql_cfg)

print(f'Training CQL: state_dim={state_dim}, K={K} actions, 500 steps (demo)')
print('(Full training uses 30,000 steps — see train.py)')

try:
    t0 = time.time()
    trainer.train(transitions)
    elapsed = time.time() - t0
    print(f'\nCQL training done in {elapsed:.1f}s')
    q_policy = trainer.get_policy()
    # Test policy on a sample state
    test_state = states[5]
    action = q_policy(test_state)
    print(f'Policy output for state[5]: bin {action} (range [0, {K-1}])')
except ImportError as e:
    print(f'd3rlpy not installed: {e}')
    print('Install with: pip install d3rlpy')
    print('Skipping CQL training demo.')
    q_policy = None
except Exception as e:
    print(f'CQL training failed: {e}')
    print('This may occur with very short panels (T=30). Use T>=100 for stable training.')
    q_policy = None

## Component 5: Ensemble Policy & Uncertainty Quantification (Stage 3)

**Section 3.3, Algorithm 1** (lines 12–13): the ensemble policy is a majority vote across P agents:

$$\hat{\pi}^{\text{ens}}(s) = \arg\max_k \sum_{p=1}^{P} \mathbf{1}\left[\hat{\pi}^{(p)}(s) = k\right]$$

When most MCMC draws agree → high posterior confidence.  
When votes are dispersed → parameter uncertainty is high; planner should be cautious.

Each MCMC draw $\theta^{(p)}$ produces a different state sequence $s_t^{(p)}$ via the counterfactual probabilities $\hat{l}^{0,(p)}_{i,t} = \sigma(\eta_{i,t}(\emptyset; \theta^{(p)}_i))$.

In [ ]:
# Cell 11 — Ensemble Policy Demonstration (using synthetic parameter perturbations)
# Note: Full MCMC ensemble requires fit_mcmc() which takes ~1 min.
# Here we demonstrate the ensemble voting mechanism with synthetic draws.
from q_ising.training.ensemble_trainer import EnsembleTrainer

print('Demonstrating ensemble majority-vote mechanism with synthetic parameter draws...')
print('(In production, draws come from MCMC posterior — see fit_mcmc() in models/ising.py)')

# Simulate vote distributions that would arise from an ensemble
rng = np.random.default_rng(42)
P = 10  # number of ensemble agents
H = 15  # demo horizon

# Simulate 'early campaign' (high agreement) vs 'late campaign' (dispersed) votes
early_votes  = np.array([8, 1, 1, 0])  # period 2: near-unanimous on bin 0
middle_votes = np.array([5, 3, 2, 0])  # period 8: moderate agreement
late_votes   = np.array([3, 3, 2, 2])  # period 15: dispersed (near saturation)

fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for ax, votes, title in zip(axes,
    [early_votes, middle_votes, late_votes],
    ['Period 2\n(Early: high agreement)', 'Period 8\n(Middle)', 'Period 15\n(Late: dispersed)']):
    ax.bar(range(K), votes, color=['#2196F3' if v == max(votes) else '#B0BEC5' for v in votes])
    ax.set_xlabel('Bin (community)')
    ax.set_ylabel('Agent votes')
    ax.set_title(title)
    ax.set_xticks(range(K))
    ax.set_ylim(0, P + 1)
    ax.axhline(P/2, color='red', linestyle='--', alpha=0.5, label='Majority threshold')

plt.suptitle(f'Ensemble Vote Distribution (P={P} agents)\nConcentrated = confident; Dispersed = uncertain',
             fontsize=11)
plt.tight_layout()
plt.savefig('../results/ensemble_votes.png', dpi=100, bbox_inches='tight')
plt.show()
print('Ensemble vote visualization saved to results/ensemble_votes.png')
print('\nThis mirrors Figure 6 in the paper (MCMC ensemble majority-vote path for Village 50).')

## Mini Evaluation: Compare Policies on the Demo Network

We compare Q-Ising against the 4 baselines from Section 5.  
**Note**: this uses the mini CQL model trained for 500 steps — results won't match the paper.
For full reproduction, run `python train.py --config configs/sbm_default.yaml`.

In [ ]:
# Cell 12 — Policy Evaluation
from q_ising.evaluation.baselines import RandomPolicy, DegreePolicy, LIRPolicy, DegreeBinPolicy
from q_ising.evaluation.metrics import PolicyEvaluator

H_TEST = 20
N_RUNS = 10  # paper uses 50; using 10 for demo speed

evaluator = PolicyEvaluator(simulator=simulator, network=network)

policies = {
    'Random':    RandomPolicy(network),
    'Degree':    DegreePolicy(network),
    'LIR':       LIRPolicy(network),
    'DegreeBin': DegreeBinPolicy(network),
}

if q_policy is not None:
    def q_ising_wrapped(y, t=0):
        s = state_ctor.build_state(y, t)
        bin_act = q_policy(s)
        members = network.get_bin_members(int(bin_act))
        return int(np.random.choice(members))
    policies['Q-Ising (500 steps)'] = q_ising_wrapped

results = evaluator.compare_policies(policies, H=H_TEST, n_runs=N_RUNS, seed=0)

# Plot adoption curves
fig, ax = plt.subplots(figsize=(9, 5))
colors = ['gray', 'blue', 'green', 'orange', 'red']
for (name, r), color in zip(results.items(), colors):
    ax.plot(r.mean_by_period, label=f'{name} ({r.mean_reward:.4f})', color=color)
    ax.fill_between(
        range(H_TEST),
        r.mean_by_period - r.std_by_period,
        r.mean_by_period + r.std_by_period,
        alpha=0.15, color=color
    )
ax.set_xlabel('Period')
ax.set_ylabel('Mean adoption rate')
ax.set_title('Adoption Curves — Mini Demo (N=60, T_train=30, 500 CQL steps)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../results/adoption_curves_demo.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 13 — Paper's Reported Results (SIR evaluation_protocol)
paper_results = {
    'experiment': 'SBM (N=500, K=4 bins, T_train=100, H=25, n_runs=50)',
    'Q-Ising':    '~0.20 (highest adoption rate, fastest ramp)',
    'Plain DQN':  '~0.15 (qualitatively similar but slower initial ramp)',
    'Degree-Bin': '~0.09 (best static topology baseline)',
    'Degree':     '~0.07',
    'LIR':        '~0.07',
    'Random':     '~0.08 (outperforms topology baselines by chance)',
    'result': 'Q-Ising matches or improves Degree-Bin in 30+/42 villages (Fig 1)',
    'key_finding': 'Ising augmentation most valuable in early campaign window',
    'village_correlation': 'Q-Ising improvement ~ -0.5 correlation with village modularity',
}

print('=' * 60)
print('PAPER REPORTED RESULTS (arXiv:2605.06564, Section 5)')
print('=' * 60)
for k, v in paper_results.items():
    print(f'  {k:20s}: {v}')
print()
print('To reproduce these results exactly, run:')
print('  python train.py --config configs/sbm_default.yaml')
print('  python train.py --config configs/village_default.yaml')

## What To Do Next

### Full Reproduction
```bash
# SBM experiment (Section 5.1) — ~2 min
python train.py --config configs/sbm_default.yaml

# Village experiment (Section 5.2) — ~20 min for all 42 villages
python data/download_villages.py        # follow instructions
python train.py --config configs/village_default.yaml

# Ensemble variant (Section 3.3) — adds ~20 min for MCMC
python train.py --config configs/sbm_default.yaml --ensemble
```

### Exploratory Analysis
Open `explore_arxiv_2605_06564.ipynb` for:
- Posterior distribution visualization (Figure 3 in paper)
- Community seeding trajectory plots (Figure 2)
- Coupling heatmaps (Figure 4)

### Implementation Assumptions (Review Before Publishing)

| Component | Assumption | Confidence |
|-----------|-----------|----------|
| EMVS solver | L1-penalized logistic regression (proxy) | **0.62** — verify against Ročková & George (2014) |
| HMC library | PyMC with NUTS | **0.65** — paper does not name the library |
| Q-network activation | ReLU | **0.80** — standard for d3rlpy |

Low-confidence sections are marked with `# WARNING: low-confidence implementation` in the source code.